Phase 6: Encoding categorical variables

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [2]:
BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/"
demo = pd.read_sas(BASE + "DEMO_L.xpt", format="xport")
cbc  = pd.read_sas(BASE + "CBC_L.xpt",  format="xport")
ghb  = pd.read_sas(BASE + "GHB_L.xpt",  format="xport")
df = demo.merge(cbc, on="SEQN").merge(ghb, on="SEQN")
print(df.shape)

(7199, 51)


In [3]:
df["DMDEDUC2"] = df["DMDEDUC2"].replace({7: np.nan, 9: np.nan})

df = df.dropna(subset=["LBXGH"])
y = (df["LBXGH"] >= 5.7).astype(int)

cbc_numeric_cols = [c for c in [
    "LBXWBCSI", "LBXLYPCT", "LBXMOPCT", "LBXNEPCT", "LBXEOPCT", "LBXBAPCT",
    "LBXRBCSI", "LBXHGB", "LBXHCT", "LBXMCVSI", "LBXMCHSI", "LBXMC",
    "LBXRDW", "LBXPLTSI", "LBXMPSI",
] if c in df.columns]

X = df[["RIAGENDR", "RIDRETH3", "DMDEDUC2"] + cbc_numeric_cols]
print(X.shape, y.mean())

(6715, 18) 0.3516008935219658


In [4]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score


class SimpleImputer(BaseEstimator, TransformerMixin):
    """Stand-in for sklearn's SimpleImputer ('median' or 'most_frequent').
    Avoids sklearn.impute, which loads a file Windows is blocking."""
    def __init__(self, strategy="median"):
        self.strategy = strategy

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.fill_ = X.median() if self.strategy == "median" else X.mode().iloc[0]
        return self

    def transform(self, X):
        return pd.DataFrame(X).fillna(self.fill_).to_numpy()

In [5]:
BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/"
demo = pd.read_sas(BASE + "DEMO_L.xpt", format="xport")   # demographics, pregnancy status
cbc  = pd.read_sas(BASE + "CBC_L.xpt",  format="xport")
ghb  = pd.read_sas(BASE + "GHB_L.xpt",  format="xport")
diq  = pd.read_sas(BASE + "DIQ_L.xpt",  format="xport")   # diabetes questionnaire
bmx  = pd.read_sas(BASE + "BMX_L.xpt",  format="xport")   # BMI

ghb = ghb.drop(columns=["WTPH2YR"], errors="ignore")      # keep one copy of the weight
df = demo.merge(cbc, on="SEQN").merge(ghb, on="SEQN")
df = df.merge(diq[["SEQN", "DIQ010", "DIQ050", "DIQ070"]], on="SEQN", how="left")
df = df.merge(bmx[["SEQN", "BMXBMI"]], on="SEQN", how="left")
print("Have CBC and HbA1c files:", len(df))

df = df[df["RIDAGEYR"] >= 20]
print("Age 20+:", len(df))

df = df[df["DIQ010"] != 1]                                 # prior diabetes diagnosis
print("No prior diabetes diagnosis:", len(df))

df = df[(df["DIQ050"] != 1) & (df["DIQ070"] != 1)]         # insulin or diabetes pills
print("Not on insulin or diabetes pills:", len(df))

df = df[df["RIDEXPRG"] != 1]                               # pregnant
print("Not pregnant:", len(df))
print("Age range:", df["RIDAGEYR"].min(), "to", df["RIDAGEYR"].max())

Have CBC and HbA1c files: 7199
Age 20+: 6064
No prior diabetes diagnosis: 5196
Not on insulin or diabetes pills: 5111
Not pregnant: 5070
Age range: 20.0 to 80.0


In [6]:
HBA1C_CUTOFF = 6.5

df["DMDEDUC2"] = df["DMDEDUC2"].replace({7: np.nan, 9: np.nan})

df = df.dropna(subset=["LBXGH", "WTPH2YR"])
df = df[df["WTPH2YR"] > 0]
print("With HbA1c and a phlebotomy weight:", len(df))

y = (df["LBXGH"] >= HBA1C_CUTOFF).astype(int)
w = df["WTPH2YR"] / df["WTPH2YR"].mean()                   # rescaled so the average weight is 1

demo_num_cols = ["RIDAGEYR", "BMXBMI"]
cbc_numeric_cols = [
    "LBXWBCSI", "LBXLYPCT", "LBXMOPCT", "LBXNEPCT", "LBXEOPCT", "LBXBAPCT",
    "LBXRBCSI", "LBXHGB", "LBXMCVSI", "LBXRDW",
    "LBXPLTSI", "LBXMPSI",
]
missing = [c for c in cbc_numeric_cols + demo_num_cols if c not in df.columns]
assert len(cbc_numeric_cols) == 12, f"Expected 12 CBC predictors, got {len(cbc_numeric_cols)}"
assert not missing, f"Not found in the data: {missing}"

X = df[demo_num_cols + ["RIAGENDR", "RIDRETH3", "DMDEDUC2"] + cbc_numeric_cols]
print("Final sample:", X.shape, "| prevalence of HbA1c >=", HBA1C_CUTOFF, ":", round(y.mean(), 3))
print("Missing BMI:", df["BMXBMI"].isna().sum())

With HbA1c and a phlebotomy weight: 4826
Final sample: (4826, 17) | prevalence of HbA1c >= 6.5 : 0.027
Missing BMI: 59


In [7]:
edu_order = [[1.0, 2.0, 3.0, 4.0, 5.0]]

def make_preprocessor(education="ordinal"):
    if education == "ordinal":
        edu = Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", OrdinalEncoder(categories=edu_order,
                                   handle_unknown="use_encoded_value",
                                   unknown_value=-1)),
        ])
    else:
        edu = Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", OneHotEncoder(handle_unknown="ignore")),
        ])

    return ColumnTransformer([
        ("demo_num", Pipeline([("imp", SimpleImputer(strategy="median")),
                               ("sc", StandardScaler())]), demo_num_cols),
        ("sex",  Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                           ("enc", OneHotEncoder(drop="if_binary"))]), ["RIAGENDR"]),
        ("race", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                           ("enc", OneHotEncoder(handle_unknown="ignore"))]), ["RIDRETH3"]),
        ("edu",  edu, ["DMDEDUC2"]),
        ("num",  Pipeline([("imp", SimpleImputer(strategy="median")),
                           ("sc", StandardScaler())]), cbc_numeric_cols),
    ])

In [8]:
def cv_auc(pipe, X, y, w, cv):
    """Returns an array of [unweighted AUC, weighted AUC] for each fold."""
    rows = []
    for tr, te in cv.split(X, y):
        m = clone(pipe)
        m.fit(X.iloc[tr], y.iloc[tr], clf__sample_weight=w.iloc[tr])
        p = m.predict_proba(X.iloc[te])[:, 1]
        rows.append([roc_auc_score(y.iloc[te], p),
                     roc_auc_score(y.iloc[te], p, sample_weight=w.iloc[te])])
    return np.array(rows)

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for edu_mode in ["ordinal", "onehot"]:
    for name, clf in [("logreg", LogisticRegression(max_iter=1000)),
                      ("rf", RandomForestClassifier(n_estimators=200, random_state=42))]:
        pipe = Pipeline([("pre", make_preprocessor(edu_mode)), ("clf", clf)])
        r = cv_auc(pipe, X, y, w, cv)
        print(f"{edu_mode:8s} {name:7s} weighted AUC = {r[:,1].mean():.3f} ± {r[:,1].std():.3f}"
              f"   (unweighted {r[:,0].mean():.3f} ± {r[:,0].std():.3f})")

ordinal  logreg  weighted AUC = 0.815 ± 0.042   (unweighted 0.790 ± 0.039)
ordinal  rf      weighted AUC = 0.762 ± 0.009   (unweighted 0.744 ± 0.018)
onehot   logreg  weighted AUC = 0.810 ± 0.048   (unweighted 0.787 ± 0.041)
onehot   rf      weighted AUC = 0.752 ± 0.019   (unweighted 0.726 ± 0.032)


## Phase 6 justification: encoding of categorical variables

**Sample and outcome.** Of 7,199 NHANES August 2021 to August 2023 participants with CBC and HbA1c files, we restricted the analysis to U.S. adults aged 20 years and older, then applied the exclusions below. The outcome is HbA1c ≥ 6.5%, with a sample prevalence of 2.7% (about 130 participants).

| Step | Excluded | Remaining |
|---|---|---|
| Participants with CBC and HbA1c files | | 7,199 |
| Aged under 20 years | 1,135 | 6,064 |
| Prior diagnosis of diabetes | 868 | 5,196 |
| Taking insulin or diabetes pills | 85 | 5,111 |
| Pregnant at the exam | 41 | 5,070 |
| Missing HbA1c value or phlebotomy weight | 244 | 4,826 |

The final sample is n = 4,826, aged 20 to 80 (NHANES records ages above 80 as 80).

**Predictors.** Age and BMI (numeric), sex, race/ethnicity and education (categorical), and the 12 measured CBC variables: white cell count, the five white cell percentages (lymphocytes, monocytes, neutrophils, eosinophils, basophils), RBC count, hemoglobin, MCV, RDW, platelet count and mean platelet volume.

**Decision.** Education (`DMDEDUC2`) is encoded as an ordinal variable (1 to 5, from "less than 9th grade" to "college graduate or above"). Sex (`RIAGENDR`) is one-hot encoded as a single 0/1 column, and race/ethnicity (`RIDRETH3`) is one-hot encoded because it is nominal. Codes 7 (refused) and 9 (don't know) in education were set to missing.

**Comparison.** We compared ordinal and one-hot encoding of education using 5-fold stratified cross-validation. Models were trained with NHANES phlebotomy weights (`WTPH2YR`, rescaled to mean 1), and ROC AUC on each held-out fold was computed with the same weights. Unweighted values are shown in parentheses.

| Education encoding | Logistic regression | Random forest |
|---|---|---|
| Ordinal | 0.815 ± 0.042 (0.790 ± 0.039) | 0.762 ± 0.009 (0.744 ± 0.018) |
| One-hot | 0.810 ± 0.048 (0.787 ± 0.041) | 0.752 ± 0.019 (0.726 ± 0.032) |

**Interpretation.** Ordinal encoding scored equal to or slightly higher than one-hot in all four comparisons, weighted and unweighted. The weighted differences are 0.005 for logistic regression and 0.010 for random forest (0.003 and 0.018 unweighted). These are of the same size as, or smaller than, the fold-to-fold variability (standard deviations of 0.009 to 0.048), so there is no clear evidence that either encoding predicts better. Weighted AUCs were about 0.02 to 0.03 higher than unweighted AUCs in all four cases. Logistic regression also scored about 0.05 to 0.06 higher than random forest with either encoding, a larger gap than any encoding difference, although this comparison is outside the scope of the encoding decision.

**Rationale for ordinal.** Because predictive performance is equivalent within the variability we observed, and ordinal was never worse, we chose ordinal on additional grounds: (1) education levels have a natural order, so one ordinal column preserves that information; (2) it adds one feature instead of five, keeping the model simple and interpretable for the eventual order/skip policy; (3) it avoids sparse indicator columns for small categories.

**Limitations.**
- Ordinal encoding assumes equal gaps between consecutive education levels. One-hot makes no such assumption. Since one-hot gave no measurable gain here, we judge that this assumption does not cost accuracy.
- With about 130 positive cases, each test fold contains roughly 26 positives, so fold AUCs are noisy. This explains the large fold standard deviations (up to 0.048), and small differences between encodings could not be detected with this design.
- About 130 positives across 17 input variables (more after one-hot encoding) is a low number of events per variable, so overfitting is a concern, especially for random forest.

**Other decisions.**
- Education is only collected for adults 20 and over, so restricting the sample to adults removes most missing education values. Any that remain are imputed with the most frequent category inside each training fold.
- Missing BMI (n = 59, 1.2%) and missing CBC values are imputed with the training-fold median.
- All encoders, imputers and scalers sit inside a scikit-learn `Pipeline`, so they are fit on the training fold only, and an unseen category in held-out data does not break the pipeline. They are fit without survey weights; only the classifiers and the AUC use the weights.
- The ± values are the standard deviation across folds, not design-based standard errors.